# TechMind — Modelo baseline de clasificación

### Equipo tejONEs

Entrena y evalúa un baseline de **TF-IDF + Regresión Logística** sobre 1.400 registros balanceados en siete categorías.

## Flujo

1. Limpia el texto con `shared.limpieza_texto.limpiar_texto`, la misma función utilizada durante la inferencia. Incluye decodificación HTML, eliminación de etiquetas, normalización y stopwords en español.
2. Divide el dataset de forma estratificada: 80 % para entrenamiento y 20 % para prueba.
3. Ajusta TF-IDF solo con entrenamiento, con un máximo de 3.000 características. Excluye números puros y conserva términos alfanuméricos como `s3`, `ipv6` y `html5`.
4. Entrena y evalúa el clasificador con precisión, *recall* y F1-score.
5. Exporta el modelo y el vectorizador, y valida predicciones sobre textos nuevos.

## Resultado

El modelo alcanzó **71 % de accuracy** y **0.71 de F1 macro**. Las categorías más fuertes fueron Mobile (F1 0.81) y Bases de Datos (F1 0.80); Backend continúa como la principal categoría a reforzar (F1 0.53).

## 1. Preparación

Carga local del dataset unificado (1.400 registros,
200 por categoría) armado en los pasos anteriores, y se usa para
entrenar el primer modelo de clasificación del proyecto.

In [1]:
from pathlib import Path
import sys
import joblib
import os
import pandas as pd

import nltk
if(nltk.download('stopwords', quiet=True)):
    print("Descarga de stopwords exitosa")

Descarga de stopwords exitosa


In [2]:
def find_project_root(start: Path) -> Path:
    """Encuentra la raíz del proyecto desde el directorio actual o sus padres."""
    for candidate in [start, *start.parents]:
        if (candidate / "data_science").exists() and (candidate / "README.md").exists():
            return candidate
    return start


base_dir = Path.cwd().resolve()
project_root = find_project_root(base_dir)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from shared.limpieza_texto import limpiar_texto #Importación de la funcion limpiar_texto del script limpieza_texto.py

CARPETA_DATA = str((project_root / "data_science" / "data").resolve())
CARPETA_PROCESADOS = str((project_root / "data_science" / "data" / "procesados").resolve())
CARPETA_MODELOS = str((project_root / "data_science" / "models").resolve())

print(f'📁 Carpeta proyecto local: {project_root.name}')

for nombre, ruta in [
    ("datos", CARPETA_DATA),
    ("datos procesados", CARPETA_PROCESADOS),
    ("modelos entrenados", CARPETA_MODELOS)
]:
    path = Path(ruta)
    if path.exists():
        print(f"✅📂 Carpeta {nombre}: {path.relative_to(project_root)}")
    else:
        print(f"❌ No se encontro la carpeta de {nombre}❌")

📁 Carpeta proyecto local: G9-LATAM-Team-25
✅📂 Carpeta datos: data_science\data
✅📂 Carpeta datos procesados: data_science\data\procesados
✅📂 Carpeta modelos entrenados: data_science\models


In [3]:
RUTA_DATASET = f'{CARPETA_PROCESADOS}/dataset_FINAL_UNIFICADO_techmind.csv'
# Carga segura con alternativa de codificación (encoding fallback) en caso de error


df = pd.read_csv(RUTA_DATASET)

print("Filas totales:", len(df))
print(df["categoria"].value_counts())

Filas totales: 1400
categoria
Backend           200
Bases de Datos    200
Cloud             200
Data Science      200
DevOps            200
Frontend          200
Mobile            200
Name: count, dtype: int64


## 2. Limpieza de texto

Se utiliza la función canónica `limpiar_texto` del módulo compartido `shared/limpieza_texto.py`, la misma que consume Backend. La función decodifica entidades HTML, elimina etiquetas y contenido no visible de `script` y `style`, convierte el texto a minúsculas, reemplaza la puntuación Unicode por espacios, normaliza los espacios y elimina las *stopwords* en español.

Compartir esta implementación garantiza que los textos de entrenamiento y los textos nuevos recibidos por la API se procesen exactamente de la misma forma.

In [4]:
df["texto_limpio"] = df["texto"].apply(limpiar_texto)

print("Ejemplo antes:")
print(df["texto"].iloc[0][:200])
print("\nEjemplo después:")
print(df["texto_limpio"].iloc[0][:200])

Ejemplo antes:
El sistema de cerraduras de puertas inteligentes basado en el concepto de Internet de las cosas con backend móvil como servicio es el desarrollo de cerraduras de puertas inteligentes respaldado por la

Ejemplo después:
sistema cerraduras puertas inteligentes basado concepto internet cosas backend móvil servicio desarrollo cerraduras puertas inteligentes respaldado tecnología computación nube almacenamiento datos mét


## 3. Vectorización TF-IDF

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

texto_entrenamiento, texto_prueba, y_entrenamiento, y_prueba = train_test_split(
    df["texto_limpio"], df["categoria"],
    test_size=0.2, random_state=42, stratify=df["categoria"]
)

vectorizador = TfidfVectorizer(
    max_features=3000,
    # Excluye números puros, pero conserva términos como s3, ipv6 y html5.
    token_pattern=r"(?u)\b(?=\w*[^\W\d_])\w{2,}\b",
)
X_entrenamiento = vectorizador.fit_transform(texto_entrenamiento)   # fit SOLO en train
X_prueba = vectorizador.transform(texto_prueba)                     # transform en test, sin fit

print("Forma de la matriz de entrenamiento (filas, columnas):", X_entrenamiento.shape)
print("Forma de la matriz de prueba (filas, columnas):", X_prueba.shape)
print("Ejemplo de palabras en el vocabulario:", vectorizador.get_feature_names_out()[:10])

Forma de la matriz de entrenamiento (filas, columnas): (1120, 3000)
Forma de la matriz de prueba (filas, columnas): (280, 3000)
Ejemplo de palabras en el vocabulario: ['2d' '2f' '3d' '5g' 'abajo' 'abierto' 'aborda' 'abordar' 'abra'
 'absoluta']


## 4. Entrenar el modelo baseline

Separar el dataset en dos partes: 80% para que el modelo aprenda (entrenamiento) y 20% para probarlo después con textos que nunca vio (prueba) — así sabemos si realmente aprendió el patrón, y no solo memorizó los ejemplos. Entrenado con Regresión Logística, un algoritmo simple.
Esto es como primera versión antes de optimizar nada. Aqui podemos aportar todos. :)

In [6]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_entrenamiento, y_entrenamiento)

print("✅ Modelo entrenado")
print("Ejemplos de entrenamiento:", X_entrenamiento.shape[0])
print("Ejemplos de prueba:", X_prueba.shape[0])

✅ Modelo entrenado
Ejemplos de entrenamiento: 1120
Ejemplos de prueba: 280


## 5. Evaluar el modelo

Se usa el 20% de datos que el modelo nunca vio para medir qué tan bien predice cada categoría. El classification_report muestra, por categoría, qué porcentaje de aciertos tuvo (métrica F1) — así se identifica si alguna categoría quedó floja y necesita más ejemplos o ajustes más adelante.

In [7]:
from sklearn.metrics import classification_report

predicciones = modelo.predict(X_prueba)

print(classification_report(y_prueba, predicciones))

                precision    recall  f1-score   support

       Backend       0.53      0.53      0.53        40
Bases de Datos       0.86      0.75      0.80        40
         Cloud       0.64      0.68      0.66        40
  Data Science       0.73      0.68      0.70        40
        DevOps       0.74      0.72      0.73        40
      Frontend       0.70      0.75      0.72        40
        Mobile       0.77      0.85      0.81        40

      accuracy                           0.71       280
     macro avg       0.71      0.71      0.71       280
  weighted avg       0.71      0.71      0.71       280



## 6. Guardar el modelo

Se guarda el modelo entrenado y el vectorizador TF-IDF en dos archivos, para que backend pueda cargarlos con joblib.load() e integrarlos a la API. ambos archivos van juntos: el vectorizador es necesario para convertir texto nuevo al mismo formato numérico con el que se entrenó el modelo.

In [ ]:
joblib.dump(modelo, f'{CARPETA_MODELOS}/modelo.pkl')
joblib.dump(vectorizador, f'{CARPETA_MODELOS}/vectorizer.pkl')

print("✅ Modelo y vectorizador guardados en:", Path(CARPETA_MODELOS).relative_to(project_root))

✅ Modelo y vectorizador guardados en: data_science\models


## 7. Prueba con un texto nuevo por categoría

Prueba con un texto que nunca nuevo —
La idea de aqui es simular lo que va a pasar cuando llegue contenido real por la API.

In [9]:
def predecir_categoria(texto, modelo, vectorizador, top_n=3):
    texto_limpio = limpiar_texto(texto)
    texto_vectorizado = vectorizador.transform([texto_limpio])

    prediccion = modelo.predict(texto_vectorizado)[0]
    probabilidades = modelo.predict_proba(texto_vectorizado)[0]
    probabilidad_maxima = max(probabilidades)
    indices_top = [
        indice
        for indice in probabilidades.argsort()[::-1]
        if modelo.classes_[indice] != prediccion
    ][:top_n]

    print("Categoría predicha:", prediccion)
    print("Probabilidad:", round(probabilidad_maxima, 2))
    print(f"\nTop {top_n} categorías siguientes:")

    top_categorias = []
    for indice in indices_top:
        categoria = modelo.classes_[indice]
        probabilidad = probabilidades[indice]
        top_categorias.append((categoria, probabilidad))
        print(f"- {categoria}: {probabilidad:.2f}")

    return {
        "categoria": prediccion,
        "probabilidad": probabilidad_maxima,
        "top_categorias": top_categorias,
    }

### Prueba 1: texto sobre DevOps

In [10]:
texto_de_prueba_devops = """
Docker es una plataforma que permite empaquetar una aplicación junto con
todas sus dependencias en un contenedor, para que funcione igual sin
importar en qué máquina se ejecute. Kubernetes se usa para orquestar
muchos contenedores Docker en producción.
"""

resultado_devops = predecir_categoria(texto_de_prueba_devops, modelo, vectorizador)

Categoría predicha: DevOps
Probabilidad: 0.28

Top 3 categorías siguientes:
- Cloud: 0.28
- Mobile: 0.12
- Frontend: 0.10


### Prueba 2: texto sobre Data Science

In [11]:
texto_prueba_data_science = """
Un equipo analiza grandes volúmenes de datos con Python y pandas para
identificar patrones de comportamiento. Después entrena un modelo de
machine learning que permite realizar predicciones y evaluar sus
resultados mediante métricas estadísticas.
"""

resultado_data_science = predecir_categoria(texto_prueba_data_science, modelo, vectorizador)

Categoría predicha: Data Science
Probabilidad: 0.45

Top 3 categorías siguientes:
- Backend: 0.13
- Bases de Datos: 0.11
- Cloud: 0.09


### Prueba 3: texto sobre Frontend

In [12]:
texto_prueba_frontend = """
La interfaz web se construye con HTML, CSS y JavaScript. React permite
crear componentes reutilizables para mostrar información en el navegador,
gestionar eventos de usuario y adaptar el diseño a teléfonos y pantallas
de diferentes tamaños.
"""

resultado_frontend = predecir_categoria(texto_prueba_frontend, modelo, vectorizador)

Categoría predicha: Frontend
Probabilidad: 0.78

Top 3 categorías siguientes:
- Backend: 0.06
- Mobile: 0.05
- Bases de Datos: 0.03


### Prueba 4: texto sobre Backend

In [13]:
texto_prueba_backend = """
El servidor expone una API REST con endpoints para autenticar usuarios,
validar solicitudes HTTP y ejecutar la lógica de negocio. La aplicación
procesa las peticiones, controla permisos y devuelve respuestas JSON al
cliente mediante servicios desarrollados en Python.
"""

resultado_backend = predecir_categoria(texto_prueba_backend, modelo, vectorizador)

Categoría predicha: Backend
Probabilidad: 0.26

Top 3 categorías siguientes:
- Cloud: 0.21
- Frontend: 0.15
- Mobile: 0.10


### Prueba 5: texto sobre Bases de Datos

In [14]:
texto_prueba_bases_datos = """
Una base de datos relacional organiza la información en tablas conectadas
mediante claves primarias y foráneas. SQL permite consultar los registros,
crear índices y ejecutar transacciones, mientras que la normalización evita
duplicados y mantiene la integridad de los datos.
"""

resultado_bases_datos = predecir_categoria(texto_prueba_bases_datos, modelo, vectorizador)

Categoría predicha: Bases de Datos
Probabilidad: 0.63

Top 3 categorías siguientes:
- Data Science: 0.10
- Cloud: 0.08
- Backend: 0.07


### Prueba 6: texto sobre Cloud

In [15]:
texto_prueba_cloud = """
La computación en la nube permite desplegar recursos bajo demanda en AWS,
Azure o Google Cloud. Los servicios cloud ofrecen máquinas virtuales,
almacenamiento escalable y funciones serverless que ajustan su capacidad
según el tráfico sin administrar infraestructura física.
"""

resultado_cloud = predecir_categoria(texto_prueba_cloud, modelo, vectorizador)

Categoría predicha: Cloud
Probabilidad: 0.74

Top 3 categorías siguientes:
- Bases de Datos: 0.06
- DevOps: 0.06
- Frontend: 0.04


### Prueba 7: texto sobre Mobile

In [16]:
texto_prueba_mobile = """
La aplicación móvil se desarrolla para teléfonos Android y iOS utilizando
Kotlin y Swift. La interfaz responde a gestos táctiles, accede a la cámara
del dispositivo y recibe notificaciones push para avisar al usuario aunque
la aplicación no esté abierta.
"""

resultado_mobile = predecir_categoria(texto_prueba_mobile, modelo, vectorizador)

Categoría predicha: Mobile
Probabilidad: 0.83

Top 3 categorías siguientes:
- Frontend: 0.04
- Backend: 0.04
- Cloud: 0.03
